In [1]:
# Explore endogeneity of agenda setting

import os
import sys
import time
import yaml
import pandas as pd
import numpy as np
import re
import subprocess

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
R_PATH = local_config['R_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

import writing_tools as wt
from utils import parse_casenum
import utils

from matplotlib import pyplot as plt
from scipy.spatial.distance import mahalanobis


plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)
with open('../../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']
EMBEDDING_DIMENSION = config['EMBEDDING_DIMENSION']

rng = np.random.default_rng(12898)

N_CLUSTERS = 3
N_COMPONENTS = 10


In [2]:
df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "ologit_regression_data.parquet"))

In [3]:
# Running R regressions

res = subprocess.run([R_PATH, LOCAL_PATH + "/src/R/20-endogenous-agenda.R"], check=True, capture_output=True, text=True)
#print(res.stdout)

In [4]:
# Load regression coefficients

coefs_df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "endogenous_agenda_coefs.parquet"))

In [5]:
# Output table

header = r"""\begin{table}[H]
\centering
\caption{Determinants of Consent Calendar and Agenda Order}
\vspace{0.2cm}
\label{tab_endogenous_agenda}
\begin{adjustbox}{max height=0.42\textheight}
\begin{threeparttable}
\begin{tabular}{lcccc}
\toprule
 & \makecell{Consent Calendar\\Binary Logit} & \makecell{Agenda Order\\OLS} \\
 & (1) & (2) \\
\midrule
 &  & \\
"""
footer = r"""\bottomrule
\end{tabular}
\begin{tablenotes}[flushleft]
\footnotesize
\item Robust standard errors in parentheses. * $p<0.1$, ** $p<0.05$, *** $p<0.01$.
\item \textit{Notes:} This table explores the endogeneity of agenda setting. Column 1 is a binary logit model predicting placement on the consent calendar. Column 2 is an OLS model predicting the order of items on the agenda, for items not on the consent calendar (which are always placed first on the agenda).
\end{tablenotes}
\end{threeparttable}
\end{adjustbox}
\end{table}
"""
reg_names = ["r3","r4"]

vars = [
    ("is_residentialTRUE", "Residential Development"),
    ("is_mixed_useTRUE", "Mixed-Use Development"),
    ("is_nonresidentialTRUE", "Non-Residential Development"),
    ("log_square_footage", "$\\ln$(Square Footage)"),
    ("height", "Height (ft)"),
    ("log2_support", "$\\log_2$(\\# Support)"),
    ("log2_support_po", "$\\log_2$(\\# Support by Public Officials)"),
    ("log2_oppose", "$\\log_2$(\\# Oppose)"),
    ("weeks_til_due", "Time to Act"),
    #("agenda_order", "Agenda Order"),
    #("num_agenda_items", "No. Agenda Items"),
    #("is_consent_calendarTRUE", "Consent Calendar"),
    ("atypicality", "Atypicality"),
]

tbl = ""
for v in vars:
    tbl += v[1] + " "
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        coef = coefs_df.loc[idx, "estimate"].values[0]
        serr = coefs_df.loc[idx, "serr"].values[0]
        stars = utils.stars(coef, serr)
        tbl += f" & {coef:.3f}$^{{{stars}}}$"
    tbl += r" \\" + "\n"
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        serr = coefs_df.loc[idx, "serr"].values[0]
        tbl += f" & ({serr:.3f})"
    tbl += r" \\ [1.8ex]" + "\n"

tbl += "\n & & & & \\\\ \n"

tbl += "Suffix Group Dummies "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="sfx_grp_CUPTRUE")
    if idx.sum()==0:
        tbl += " & N "
    else:
        tbl += " & Y "
tbl += r" \\ " + "\n"

tbl += "Council District Dummies "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="cd_1")
    if idx.sum()==0:
        tbl += " & N "
    else:
        tbl += " & Y "
tbl += r" \\ " + "\n"

tbl += "Year Dummies "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="yr_2019")
    if idx.sum()==0:
        tbl += " & N "
    else:
        tbl += " & Y "
tbl += r" \\ " + "\n"

tbl += "Embedding Cluster Dummies "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="cluster_fe1TRUE")
    if idx.sum()==0:
        tbl += " & N "
    else:
        tbl += " & Y "
tbl += r" \\ " + "\n"



tbl += "\n & & \\\\ \n"

tbl += "Observations "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="num_obs")
    nobs = coefs_df.loc[idx, "estimate"].values[0]
    tbl += f" & {nobs:,.0f}"
tbl += r" \\ [1.8ex]" + "\n"

table_tex = header + tbl + footer    

with open(os.path.join(LOCAL_PATH, "tables", "tab_endogenous_agenda.tex"), "w") as f:
    f.write(table_tex)

print(table_tex)

\begin{table}[H]
\centering
\caption{Determinants of Consent Calendar and Agenda Order}
\vspace{0.2cm}
\label{tab_endogenous_agenda}
\begin{adjustbox}{max height=0.42\textheight}
\begin{threeparttable}
\begin{tabular}{lcccc}
\toprule
 & \makecell{Consent Calendar\\Binary Logit} & \makecell{Agenda Order\\OLS} \\
 & (1) & (2) \\
\midrule
 &  & \\
Residential Development  & -1.435$^{*}$ & 0.355$^{}$ \\
 & (0.766) & (0.367) \\ [1.8ex]
Mixed-Use Development  & -0.604$^{}$ & 0.504$^{}$ \\
 & (0.792) & (0.377) \\ [1.8ex]
Non-Residential Development  & 0.044$^{}$ & 0.006$^{}$ \\
 & (0.629) & (0.377) \\ [1.8ex]
$\ln$(Square Footage)  & 0.000$^{}$ & 0.049$^{}$ \\
 & (0.113) & (0.067) \\ [1.8ex]
Height (ft)  & -0.002$^{}$ & -0.000$^{}$ \\
 & (0.002) & (0.001) \\ [1.8ex]
$\log_2$(\# Support)  & -0.745$^{***}$ & 0.082$^{}$ \\
 & (0.184) & (0.065) \\ [1.8ex]
$\log_2$(\# Support by Public Officials)  & 0.232$^{}$ & -0.019$^{}$ \\
 & (0.674) & (0.244) \\ [1.8ex]
$\log_2$(\# Oppose)  & -0.574$^{***}$ &